In [ ]:
import os
import json as _json
from pathlib import Path
from dataclasses import dataclass, field
from typing import cast
from typing import List, Optional, Dict, Any, Tuple, Union, Callable, Set
import pandas as pd
import polars as pl
import io
import numpy as np
from types import SimpleNamespace
from polars.testing import assert_frame_equal as pl_assert_frame_equal
from itertools import product
DataFrame = pd.DataFrame
print('pandas:', pd.__version__, ' polars:', pl.__version__)
ModelConfig = object


In [ ]:
# ── Fixtures ────────────────────────────────────────────────────────────────

# --- src_onemod_utils_parameters_create_params_migration ---
FIX_SRC_ONEMOD_UTILS_PARAMETERS_CREATE_PARAMS_MIGRATION_EXTRACT_PARAM_DICT = lambda config: {
    "alpha": [0.1, 0.2],
    "beta": ["low", "high"],
}
FIX_SRC_ONEMOD_UTILS_PARAMETERS_CREATE_PARAMS_EMPTY = lambda config: {}

# --- src_onemod_utils_parameters_get_params_migration ---
FIX_SRC_ONEMOD_UTILS_PARAMETERS_GET_PARAMS_PD = pd.DataFrame({"param_id": [0, 1], "alpha": [0.1, 0.2], "beta": ["low", "high"]})
FIX_SRC_ONEMOD_UTILS_PARAMETERS_GET_PARAMS_PL = pl.from_pandas(FIX_SRC_ONEMOD_UTILS_PARAMETERS_GET_PARAMS_PD)

try:
    pd.Index.__class_getitem__
except AttributeError:
    pd.Index.__class_getitem__ = classmethod(lambda cls, item: cls)

print("✅ Fixtures loaded")


In [ ]:
# ── Before wrappers (verbatim pandas) ───────────────────────────────────────

def before_src_onemod_utils_parameters_create_params_migration(extract_param_dict):
    def create_params(config: ModelConfig) -> DataFrame | None:
        param_dict = extract_param_dict(config)
        if len(param_dict) == 0:
            return None
        crossby = list(param_dict.keys())
        params = DataFrame(
            [param_set for param_set in product(*param_dict.values())],
            columns=crossby,
        )
        params["param_id"] = params.index
        return params[["param_id", *crossby]]
    return create_params

def before_src_onemod_utils_parameters_get_params_migration():
    def get_params(params: DataFrame, param_id: int) -> dict[str, Any]:
        params = params.query("param_id == @param_id").drop(columns=["param_id"])
        return {
            str(param_name): param_value.item()
            for param_name, param_value in params.items()
        }
    return get_params

In [ ]:
# ── Generated wrappers (experiment-generated Polars) ─────────────────────────

def gen_src_onemod_utils_parameters_create_params_migration(extract_param_dict):
    from itertools import product



    def create_params(config: ModelConfig) -> pl.DataFrame | None:
        param_dict = extract_param_dict(config)
        if len(param_dict) == 0:
            return None
        crossby = list(param_dict.keys())
        params = pl.DataFrame(
            [param_set for param_set in product(*param_dict.values())],
            schema=crossby,
        )
        params = params.with_row_index("param_id")
        return params.select(["param_id", *crossby])
    return create_params

def gen_src_onemod_utils_parameters_get_params_migration():
    import polars as pl
    from polars import DataFrame


    def get_params(params: DataFrame, param_id: int) -> dict[str, Any]:
        params = params.filter(pl.col("param_id") == param_id).drop("param_id")
        return {
            str(param_name): param_value
            for param_name, param_value in params.row(0, named=True).items()
        }
    return get_params

In [ ]:
# ── Comparison helper ───────────────────────────────────────────────────────
def _index_is_trivial(idx):
    # Unnamed + integer-valued covers both a fresh RangeIndex and the leftover
    # positional index after filtering/boolean-masking a RangeIndex-based frame
    # (pandas downgrades RangeIndex to a plain Int64Index on filter, but it's
    # still just leftover row positions, not real data). A set_index(...)
    # always carries the original column's name, so any genuinely meaningful
    # index is caught by the "name is not None" branch.
    return idx.name is None and pd.api.types.is_integer_dtype(idx.dtype)


def _to_pl(r):
    if isinstance(r, pl.DataFrame): return r
    if isinstance(r, pd.DataFrame): return pl.from_pandas(r.reset_index(drop=True) if _index_is_trivial(r.index) else r.reset_index())
    if isinstance(r, pd.Series): return pl.from_pandas(r.to_frame().reset_index(drop=True) if _index_is_trivial(r.index) else r.to_frame().reset_index())
    return None

def compare(before_result, gen_result, label, check_row_order=False):
    raw_label = str(label)
    label_parts = raw_label.strip().split()
    is_l3 = bool(label_parts and label_parts[0].upper() == "L3")
    layer = "L3" if is_l3 else "L2"
    kind = "edge" if is_l3 else "equivalence"
    if is_l3:
        label_parts = label_parts[1:]
        if label_parts and label_parts[0].lower() in ("edge", "branch"):
            label_parts = label_parts[1:]
        display_label = " ".join(label_parts)
    else:
        display_label = raw_label

    left  = _to_pl(before_result.collect() if isinstance(before_result, pl.LazyFrame) else before_result)
    right = _to_pl(gen_result.collect() if isinstance(gen_result, pl.LazyFrame) else gen_result)
    if left is None and right is None:
        print(f"⚠️  {layer} {kind} {display_label}: both sides non-DataFrame (no output to compare)")
        return
    if left is None or right is None:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — one side returned DataFrame, other did not")
        return
    left_cols, right_cols = set(left.columns), set(right.columns)
    if left_cols != right_cols:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — column sets differ (before-only={left_cols - right_cols}, gen-only={right_cols - left_cols})")
        return
    common = list(left.columns)
    try:
        pl_assert_frame_equal(left.select(common), right.select(common),
                              check_dtypes=False, check_row_order=check_row_order)
        print(f"✅ {layer} {kind} {display_label}: MATCH")
    except Exception as e:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — {e}")


In [ ]:
# === Tests: src_onemod_utils_parameters_create_params_migration ===

try:
    _create = gen_src_onemod_utils_parameters_create_params_migration(FIX_SRC_ONEMOD_UTILS_PARAMETERS_CREATE_PARAMS_MIGRATION_EXTRACT_PARAM_DICT)
    _r = _create(SimpleNamespace())
    print("✅ L1 smoke gen_src_onemod_utils_parameters_create_params_migration: OK, type=", type(_r).__name__)
except Exception as _e:
    print(f"❌ L1 smoke gen_src_onemod_utils_parameters_create_params_migration: {type(_e).__name__}: {_e}")

try:
    _create = before_src_onemod_utils_parameters_create_params_migration(FIX_SRC_ONEMOD_UTILS_PARAMETERS_CREATE_PARAMS_MIGRATION_EXTRACT_PARAM_DICT)
    _rb = _create(SimpleNamespace())
    print("✅ L1 smoke before_src_onemod_utils_parameters_create_params_migration: OK")
except Exception as _e:
    print(f"❌ L1 smoke before_src_onemod_utils_parameters_create_params_migration: {type(_e).__name__}: {_e}")

try:
    _rb = before_src_onemod_utils_parameters_create_params_migration(FIX_SRC_ONEMOD_UTILS_PARAMETERS_CREATE_PARAMS_MIGRATION_EXTRACT_PARAM_DICT)(SimpleNamespace())
    _rg = gen_src_onemod_utils_parameters_create_params_migration(FIX_SRC_ONEMOD_UTILS_PARAMETERS_CREATE_PARAMS_MIGRATION_EXTRACT_PARAM_DICT)(SimpleNamespace())
    compare(_rb, _rg, "src_onemod_utils_parameters_create_params_migration")
except Exception as _e:
    print(f"❌ L2 equivalence src_onemod_utils_parameters_create_params_migration: setup error — {type(_e).__name__}: {_e}")

try:
    _rb = before_src_onemod_utils_parameters_create_params_migration(FIX_SRC_ONEMOD_UTILS_PARAMETERS_CREATE_PARAMS_EMPTY)(SimpleNamespace())
    _rg = gen_src_onemod_utils_parameters_create_params_migration(FIX_SRC_ONEMOD_UTILS_PARAMETERS_CREATE_PARAMS_EMPTY)(SimpleNamespace())
    print("✅ L3 edge src_onemod_utils_parameters_create_params_migration empty dict: MATCH" if _rb is None and _rg is None else f"❌ L3 edge src_onemod_utils_parameters_create_params_migration empty dict: MISMATCH — before={_rb}, gen={_rg}")
except Exception as _e:
    print(f"❌ L3 edge src_onemod_utils_parameters_create_params_migration: {type(_e).__name__}: {_e}")
